# Campos Gerais Water: Reproducing Published Runoff Contrasts

Project lead: Lissandra Kruse Fuganti.

Source: de Paula et al. (2026), https://doi.org/10.1007/s11368-026-04517-0, Supplementary Table S1, CC BY 4.0. Attributed transcription converted to CSV; these measurements were not collected by this project author. Code developed with AI assistance; human review remains necessary.

**24 selected runoff-event records, 20 dates, 96 rounded percentage contrasts. Not raw time series, not 24 independent sites, and not all eligible rainfall events. No machine learning, causal inference, waterlogging prediction or regional validation. Unweighted medians are not pooled reductions. 100% is interpreted only as reported.**

This self-contained notebook requires only the Python standard library: no network, GPU, credentials or installations. Run all cells to generate JSON, CSV, a report and a SHA-256 manifest.

Repository: https://github.com/lissandrakruse/campos-gerais-agua


In [ ]:
from pathlib import Path
import sys
ROOT = Path("/kaggle/working/campos-gerais-agua") if Path("/kaggle/working").exists() else Path("campos-gerais-agua-notebook")
ROOT.mkdir(parents=True, exist_ok=True)
p=ROOT/'scripts/extract_springer_s1.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_bytes(b'#!/usr/bin/env python3\n"""Extract published derived contrasts, NOT raw runoff, from de Paula et al. S1.\nInput: official DOCX; output: sourced CSV and descriptive audit (no inference).\n"""\nimport argparse,csv,datetime,hashlib,json,re,statistics,zipfile\nfrom pathlib import Path\nimport xml.etree.ElementTree as ET\nDOI=\'10.1007/s11368-026-04517-0\'\nURL=\'https://media.springernature.com/original/springer-static/esm/art%3A10.1007%2Fs11368-026-04517-0/MediaObjects/11368_2026_4517_MOESM1_ESM.docx\'\nMETRICS=[\'ssc_med_difference_pct\',\'ssc_max_difference_pct\',\'trv_difference_pct\',\'tsy_difference_pct\']\nNS={\'w\':\'http://schemas.openxmlformats.org/wordprocessingml/2006/main\'}\ndef extract(path):\n root=ET.fromstring(zipfile.ZipFile(path).read(\'word/document.xml\'))\n tables=root.findall(\'.//w:tbl\',NS)\n assert len(tables)==1, \'Review changed source: expected one table\'\n rows=[]\n for tr in tables[0].findall(\'w:tr\',NS)[3:]:\n  cells=[\'\'.join(t.itertext()).strip() for t in tr.findall(\'w:tc\',NS)]\n  assert len(cells)==5\n  label=cells[0];datepart,_,suffix=label.partition(\'_\')\n  date=datetime.datetime.strptime(datepart,\'%b %d, %Y\').date().isoformat()\n  row={\'source_row\':len(rows)+1,\'event_label_original\':label,\'event_date\':date,\'event_suffix\':suffix}\n  for name,value in zip(METRICS,cells[1:]): row[name]=float(value.replace(\',\',\'.\'))\n  row[\'source_doi\']=DOI;row[\'source_table\']=\'Supplementary Table S1\';rows.append(row)\n assert len(rows)==24\n return rows\ndef describe(rows):\n dates={r[\'event_date\'] for r in rows}\n out={\'n_event_rows\':len(rows),\'n_distinct_dates\':len(dates),\'duplicate_full_event_labels\':len(rows)-len({r[\'event_label_original\'] for r in rows}), \'multiple_events_same_date\':{d:sum(r[\'event_date\']==d for r in rows) for d in sorted(dates) if sum(r[\'event_date\']==d for r in rows)>1},\'year_counts\':{y:sum(r[\'event_date\'][:4]==y for r in rows) for y in sorted({r[\'event_date\'][:4] for r in rows})},\'metrics\':{}}\n for m in METRICS:\n  a=[r[m] for r in rows];out[\'metrics\'][m]={\'n\':len(a),\'min_pct\':min(a),\'max_pct\':max(a),\'median_pct_unweighted\':statistics.median(a),\'n_negative\':sum(x<0 for x in a),\'n_zero\':sum(x==0 for x in a),\'n_positive\':sum(x>0 for x in a),\'n_equal_100_reported\':sum(x==100 for x in a)}\n out[\'interpretation\']=\'Published rounded relative contrasts only. No absolute runoff volumes, rainfall predictors or sampling denominators. Unweighted descriptions are not pooled reductions, confidence intervals or independent treatment effects. 100% is as reported, not independently verified absence.\'\n return out\ndef main():\n p=argparse.ArgumentParser();p.add_argument(\'docx\',type=Path);p.add_argument(\'--out\',type=Path,default=Path(\'.\'));a=p.parse_args();a.out.mkdir(parents=True,exist_ok=True)\n rows=extract(a.docx)\n with (a.out/\'de_paula_2026_s1_derived_event_contrasts.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n  w=csv.DictWriter(f,fieldnames=list(rows[0]),lineterminator="\\n");w.writeheader();w.writerows(rows)\n summary=describe(rows);summary[\'source_sha256\']=hashlib.sha256(a.docx.read_bytes()).hexdigest();summary[\'source_url\']=URL\n (a.out/\'de_paula_2026_s1_audit.json\').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+\'\\n\')\n print(json.dumps(summary,indent=2))\nif __name__==\'__main__\':main()\n')
p=ROOT/'scripts/run_pipeline.py'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_bytes(b'#!/usr/bin/env python3\n"""Deterministic descriptive pipeline for the published S1 contrasts only."""\nimport argparse, csv, datetime, hashlib, json, math\nfrom pathlib import Path\nfrom extract_springer_s1 import describe, METRICS, DOI, URL\nEXPECTED_CSV_SHA256 = "52ff69ca53da06c2bd2d441629b79efe64a5435973d9c1957f97096cfbf10215"\nDEFAULT_INPUT = Path(__file__).resolve().parents[1] / "data/published_derived/de_paula_2026_s1_derived_event_contrasts.csv"\nCAVEAT = "Published rounded percentage contrasts, not raw measurements. Unweighted medians are not pooled reductions. No predictive model, causal inference, or regional validation. These 24 selected runoff-event records cover 20 dates of one experiment, not 24 independent sites or all eligible storms. Reported 100% is not independently verified absence. Outlet runoff is not within-field waterlogging. Occurrence classification requires all eligible storms, including verified zero-runoff events."\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\ndef validate(path):\n    # Exact bytes pin all 96 numeric cells, event labels and attribution to audited version.\n    if sha(path) != EXPECTED_CSV_SHA256:\n        raise ValueError("CSV differs from the audited published table; re-audit required")\n    with Path(path).open(encoding="utf-8", newline="") as f:\n        rows = list(csv.DictReader(f))\n    if len(rows) != 24:\n        raise ValueError("Expected 24 source rows")\n    for i, row in enumerate(rows, 1):\n        if int(row["source_row"]) != i or row["source_doi"] != DOI:\n            raise ValueError("Invalid source provenance")\n        datetime.date.fromisoformat(row["event_date"])\n        row["source_row"] = i\n        for m in METRICS:\n            row[m] = float(row[m])\n            if not math.isfinite(row[m]):\n                raise ValueError("Nonfinite source value")\n    return rows\n\ndef run(input_path=DEFAULT_INPUT, out=Path("outputs/published")):\n    input_path, out = Path(input_path), Path(out)\n    rows = validate(input_path)\n    summary = describe(rows)\n    out.mkdir(parents=True, exist_ok=True)\n    (out / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True)+"\\n", encoding="utf-8")\n    fields = ["metric"] + list(summary["metrics"][METRICS[0]])\n    with (out / "metrics.csv").open("w", encoding="utf-8", newline="") as f:\n        writer=csv.DictWriter(f,fieldnames=fields); writer.writeheader()\n        for m in METRICS: writer.writerow({"metric":m, **summary["metrics"][m]})\n    lines=["# Descriptive reproduction of Supplementary Table S1", "", CAVEAT, "", f"Source: https://doi.org/{DOI} (Supplementary Table S1; CC BY 4.0).", "", "24 event rows, 20 dates, 96 percentage values checked against the audited version.", "", "| Metric | Minimum (%) | Maximum (%) | Unweighted median (%) | Negative values |", "|---|---:|---:|---:|---:|"]\n    for m in METRICS:\n        s=summary["metrics"][m]\n        lines.append(f"| {m} | {s[\'min_pct\']} | {s[\'max_pct\']} | {s[\'median_pct_unweighted\']} | {s[\'n_negative\']} |")\n    lines += ["", "No pooled reduction is calculated: absolute denominators are missing. No confidence intervals or machine-learning performance are calculated. Original extraction and limitations are documented in data/published_derived/PROVENANCE.md and docs/dados.md.", "", "Row counts do not measure spatial replication. Date inconsistencies between papers are not automatically reconciled. Future limited models may use audited event aggregates and predictors; minute-level hydrographs are not required for every model."]\n    (out / "report.md").write_text("\\n".join(lines)+"\\n",encoding="utf-8")\n    manifest={"pipeline_version":"1.0.0", "input_sha256":sha(input_path), "source_doi":DOI,"source_url":URL,"license_derived_table":"CC BY 4.0", "mode":"published_derived_only", "outputs":{n:sha(out/n) for n in ["summary.json","metrics.csv","report.md"]}}\n    # No wall-clock timestamps or absolute paths: same bytes in every output directory.\n    (out / "manifest.json").write_text(json.dumps(manifest,indent=2,sort_keys=True)+"\\n",encoding="utf-8")\n    return summary\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--input", type=Path, default=DEFAULT_INPUT)\n    p.add_argument("--out", type=Path, default=Path("outputs/published"))\n    a=p.parse_args(); s=run(a.input,a.out)\n    print(f"Validated {s[\'n_event_rows\']} event rows; descriptive outputs: {a.out}")\nif __name__ == "__main__": main()\n')
p=ROOT/'data/published_derived/de_paula_2026_s1_derived_event_contrasts.csv'
p.parent.mkdir(parents=True,exist_ok=True)
p.write_bytes(b'source_row,event_label_original,event_date,event_suffix,ssc_med_difference_pct,ssc_max_difference_pct,trv_difference_pct,tsy_difference_pct,source_doi,source_table\n1,"Feb 17, 2023",2023-02-17,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n2,"Mar 2, 2023_T",2023-03-02,T,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n3,"Mar 3, 2023_N",2023-03-03,N,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n4,"Mar 3, 2023",2023-03-03,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n5,"Apr 16, 2023",2023-04-16,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n6,"Apr 18, 2023",2023-04-18,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n7,"Jul 27, 2023",2023-07-27,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n8,"Aug 11, 2023",2023-08-11,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n9,"Sep 2, 2023",2023-09-02,,88.04,87.8,57.14,99.82,10.1007/s11368-026-04517-0,Supplementary Table S1\n10,"Oct 8, 2023",2023-10-08,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n11,"Oct 12, 2023_T",2023-10-12,T,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n12,"Oct 12, 2023_N",2023-10-12,N,5.19,35.6,64.29,80.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n13,"Oct 26, 2023",2023-10-26,,100.0,100.0,96.55,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n14,"Oct 27, 2023",2023-10-27,,-18.24,-27.82,60.09,42.5,10.1007/s11368-026-04517-0,Supplementary Table S1\n15,"Oct 28, 2023_T",2023-10-28,T,67.51,98.5,85.24,95.65,10.1007/s11368-026-04517-0,Supplementary Table S1\n16,"Oct 28, 2023_N",2023-10-28,N,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n17,"Oct 29, 2023",2023-10-29,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n18,"Oct 30, 2023",2023-10-30,,46.38,-62.28,81.44,87.87,10.1007/s11368-026-04517-0,Supplementary Table S1\n19,"Dec 4, 2023",2023-12-04,,100.0,100.0,100.0,100.0,10.1007/s11368-026-04517-0,Supplementary Table S1\n20,"Nov 28, 2024",2024-11-28,,-118.93,-14.72,41.35,-9.66,10.1007/s11368-026-04517-0,Supplementary Table S1\n21,"Dec 7, 2024",2024-12-07,,13.68,-30.68,35.99,64.72,10.1007/s11368-026-04517-0,Supplementary Table S1\n22,"Dec 9, 2024_T1",2024-12-09,T1,41.62,55.94,25.89,-347.22,10.1007/s11368-026-04517-0,Supplementary Table S1\n23,"Dec 9, 2024_T2",2024-12-09,T2,-35.46,-8.63,44.17,27.22,10.1007/s11368-026-04517-0,Supplementary Table S1\n24,"Jan 31, 2025",2025-01-31,,-53.43,-172.21,30.74,-2.27,10.1007/s11368-026-04517-0,Supplementary Table S1\n')
sys.path.insert(0,str(ROOT/"scripts"))
from run_pipeline import run
summary=run(ROOT/"data/published_derived/de_paula_2026_s1_derived_event_contrasts.csv",ROOT/"outputs/published")
print((ROOT/"outputs/published/report.md").read_text())


In [ ]:
import json
manifest=json.loads((ROOT/"outputs/published/manifest.json").read_text())
print(json.dumps(manifest,indent=2))


## Limitations and next steps

The pipeline pins the audited CSV with a hash; it does not independently validate instruments or the original experiment. For a fresh extraction, obtain the official DOCX through the DOI supplementary-material link and run the repository extractor. Changed input requires a fresh audit. The CSV changes decimal commas to decimal points and adds ISO dates while preserving source labels and attribution.

The contrasts are reported as 100 × (NT − NT+T) / NT, comparing non-terraced and terraced no-till plots. Negative values are retained. Absolute runoff volumes and rainfall predictors cannot be reconstructed from percentages alone. Same-date events and both plots cannot be treated as independent validation samples. Date inconsistencies across papers require review before joining sources.

These are selected runoff records. Occurrence classification would need the full eligible storm set, including verified zero-runoff events. Audited event aggregates with predictors may support a limited future model; minute-level hydrographs are not mandatory for every model. Independent validation is still required. Post-event vegetation images cannot serve as pre-event predictors.

Synthetic examples belong to a separate schema-validation flow and are rejected here. No operational irrigation recommendations are made. The CC BY 4.0 attribution applies to the derived source table; it is not a blanket license for every project artifact.
